# 📊 AI-Assisted Trading Risk Manager
**Sentiment → Risk Models → Portfolio Optimisation → Trailing Stops → Monte Carlo**

**Install dependencies:**
```bash
pip install yfinance transformers torch numpy scipy plotly arch hmmlearn
```

> **All tunable parameters live in the two config cells below.**  
> You should rarely need to touch anything else.


## ⚙️ Master Configuration
*Edit anything here — every downstream cell reads from these variables.*

In [14]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# 1. PORTFOLIO
# ─────────────────────────────────────────────────────────────────────────────
TICKERS   = ['AAPL', 'MSFT', 'XOM', 'GS', 'JPM']
START     = '2020-01-01'
END       = '2026-05-22'
RISK_FREE = 0.05     # annual risk-free rate (decimal)

# ── Portfolio weighting ───────────────────────────────────────────────────
# 'equal'     → 1/N weight on every ticker
# 'manual'    → use MANUAL_WEIGHTS below
# 'optimised' → use max-Sharpe weights from Phase 3
#               (run Phase 3 first, then re-run Phase 2b onward)
PORTFOLIO_WEIGHTING = 'equal'

# Used only when PORTFOLIO_WEIGHTING = 'manual'. Must sum to 1.
MANUAL_WEIGHTS = {
    'AAPL': 0.30,
    'MSFT': 0.25,
    'XOM':  0.20,
    'GS':   0.15,
    'JPM':  0.10,
}

# ─────────────────────────────────────────────────────────────────────────────
# 2. SENTIMENT
# ─────────────────────────────────────────────────────────────────────────────
MAX_HEADLINES_PER_TICKER = 6

# ─────────────────────────────────────────────────────────────────────────────
# 3. RISK MODELS  (all run on the weighted portfolio return series)
# ─────────────────────────────────────────────────────────────────────────────
ROLLING_WINDOW        = 21             # days for rolling vol window
EWMA_SPAN             = 21             # EWMA span (λ ≈ 1 - 2/(span+1))
GARCH_P               = 1
GARCH_Q               = 1
VAR_CONFIDENCE_LEVELS = (0.95, 0.99)
VAR_SIM_SIZE          = 100_000

# ─────────────────────────────────────────────────────────────────────────────
# 4. REGIME DETECTION (HMM) — runs on portfolio returns
# ─────────────────────────────────────────────────────────────────────────────
HMM_N_REGIMES = 3
HMM_N_ITER    = 200

# ─────────────────────────────────────────────────────────────────────────────
# 5. PORTFOLIO OPTIMISATION
# ─────────────────────────────────────────────────────────────────────────────
N_SIM_EF               = 3_000
SENTIMENT_RETURN_BOOST = 0.01   # expected-return nudge per unit of sentiment

# ─────────────────────────────────────────────────────────────────────────────
# 6. MONTE CARLO
# ─────────────────────────────────────────────────────────────────────────────
HORIZON               = 252
N_PATHS               = 10_000
PORT_VAL              = 1_000_000   # starting portfolio value ($)
RANDOM_SEED           = 42
PLOT_SAMPLE_PATHS     = 200
SENTIMENT_DRIFT_NUDGE = 0.0002      # daily drift nudge per unit of sentiment

# Regime GBM parameters — ANNUAL drift & vol (converted to daily inside MC)
REGIME_PARAMS = {
    'Bull 🟢':    {'drift':  0.12, 'vol': 0.12},
    'Neutral ⚪': {'drift':  0.04, 'vol': 0.18},
    'Bear 🔴':    {'drift': -0.08, 'vol': 0.28},
}

# ─────────────────────────────────────────────────────────────────────────────
# 7. STRESS TEST
# ─────────────────────────────────────────────────────────────────────────────
N_PATHS_STRESS = 5_000

# Each scenario adjusts two GBM parameters relative to the base regime:
#   drift_shock  — annual return change (e.g. -0.10 = loses 10% annual return)
#   vol_mult     — multiplier on base vol  (e.g.  2.0 = twice as volatile)
# See the "How to set scenarios" markdown cell below for guidance.
SCENARIOS = {
    'Base case':         {'drift_shock':  0.00, 'vol_mult': 1.0},
    'Oil −20%':          {'drift_shock': -0.06, 'vol_mult': 1.4},
    'Rates +100 bps':    {'drift_shock': -0.03, 'vol_mult': 1.2},
    'Market crash −30%': {'drift_shock': -0.25, 'vol_mult': 2.5},
}

print('✅ Master config loaded.')


✅ Master config loaded.


## ⚙️ Trailing Stop Configuration
Each position is split into **3 tranches** (⅓ each).  
A stop watches the **drawdown from the running peak**; when breached, that tranche locks in as cash.

| Mode | How to activate |
|---|---|
| **Sentiment-derived** *(default)* | Keep `USE_MANUAL = False` — levels auto-computed after Phase 1 |
| **Manual override** | Set `USE_MANUAL = True` and fill `MANUAL_STOP_LEVELS` / `_FRACTIONS` |


In [15]:
USE_MANUAL = False

# ── Manual override ───────────────────────────────────────────────────────
MANUAL_STOP_LEVELS    = [-0.05, -0.10, -0.15]  # drawdown from peak (negative %)
MANUAL_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]  # fraction of remaining position to exit

# ── Sentiment thresholds ──────────────────────────────────────────────────
SENTIMENT_BULL_THRESHOLD =  0.3
SENTIMENT_BEAR_THRESHOLD = -0.3

# ── Auto-derived levels per sentiment regime ──────────────────────────────
BULL_STOP_LEVELS    = [-0.08, -0.14, -0.20]   # wide — let winners run
BULL_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]

NEUTRAL_STOP_LEVELS    = [-0.05, -0.10, -0.15]
NEUTRAL_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]

BEAR_STOP_LEVELS    = [-0.03, -0.06, -0.10]   # tight — protect capital
BEAR_STOP_FRACTIONS = [1/3,   1/3,   1/3  ]

def build_stops(sentiment_score):
    if USE_MANUAL:
        levels, fracs, tag = MANUAL_STOP_LEVELS, MANUAL_STOP_FRACTIONS, '⚙️  Manual'
    elif sentiment_score >= SENTIMENT_BULL_THRESHOLD:
        levels, fracs, tag = BULL_STOP_LEVELS, BULL_STOP_FRACTIONS, '🟢 Bullish-derived'
    elif sentiment_score <= SENTIMENT_BEAR_THRESHOLD:
        levels, fracs, tag = BEAR_STOP_LEVELS, BEAR_STOP_FRACTIONS, '🔴 Bearish-derived'
    else:
        levels, fracs, tag = NEUTRAL_STOP_LEVELS, NEUTRAL_STOP_FRACTIONS, '⚪ Neutral-derived'
    return ([{'level': l, 'exit_fraction': f, 'label': f'Stop {i+1} ({l:.0%})'}
             for i, (l, f) in enumerate(zip(levels, fracs))], tag)

print('✅ Trailing stop config loaded. Run Phase 1 to finalise levels.')


✅ Trailing stop config loaded. Run Phase 1 to finalise levels.


---
## Phase 1 — Financial Sentiment Engine (FinBERT + live news)
Fetches real headlines for every ticker via `yfinance`. Falls back to sample headlines if unavailable.


In [16]:
from transformers import pipeline

def get_ticker_headlines(tickers, max_per=MAX_HEADLINES_PER_TICKER):
    results = []
    for t in tickers:
        try:
            news = yf.Ticker(t).news or []
            for item in news[:max_per]:
                title = (item.get('content', {}).get('title')
                         if isinstance(item.get('content'), dict)
                         else item.get('title', ''))
                if title:
                    results.append({'ticker': t, 'headline': title})
        except Exception as e:
            print(f'  ⚠️  {t}: {e}')
    return results

print('Fetching live headlines...')
live_items = get_ticker_headlines(TICKERS)

FALLBACK = [
    {'ticker': 'AAPL', 'headline': 'Apple beats earnings expectations, raises guidance'},
    {'ticker': 'AAPL', 'headline': 'iPhone demand stronger than anticipated in Asia'},
    {'ticker': 'MSFT', 'headline': 'Microsoft Azure cloud revenue surges 33% year-on-year'},
    {'ticker': 'MSFT', 'headline': 'Goldman Sachs upgrades Microsoft to strong buy'},
    {'ticker': 'XOM',  'headline': 'Oil prices tumble on demand fears amid global slowdown'},
    {'ticker': 'XOM',  'headline': 'ExxonMobil raises dividend as oil profits remain elevated'},
    {'ticker': 'GS',   'headline': 'Goldman Sachs Q3 profit rises on trading revenue rebound'},
    {'ticker': 'GS',   'headline': 'Wall Street banks face tighter capital requirements'},
    {'ticker': 'JPM',  'headline': 'JPMorgan warns of credit losses in commercial real estate'},
    {'ticker': 'JPM',  'headline': 'Fed signals two more rate hikes this year'},
]

headline_items = live_items if live_items else FALLBACK
if not live_items:
    print('⚠️  No live data — using fallback headlines.')
else:
    print(f'✅ {len(live_items)} live headlines fetched.')

headlines = [h['headline'] for h in headline_items]

print('\nLoading FinBERT (~420 MB on first run)...')
sentiment_pipe = pipeline('text-classification', model='ProsusAI/finbert', top_k=None)
results = sentiment_pipe(headlines)

rows = []
for item, scores in zip(headline_items, results):
    if isinstance(scores, dict): scores = [scores]
    sd = {s['label']: s['score'] for s in scores}
    rows.append({'ticker': item['ticker'], 'headline': item['headline'],
                 'positive': round(sd.get('positive',0),3),
                 'negative': round(sd.get('negative',0),3),
                 'neutral':  round(sd.get('neutral', 0),3),
                 'sentiment': max(sd, key=sd.get)})

sentiment_df = pd.DataFrame(rows)
display(sentiment_df)

ticker_sentiment = (sentiment_df.groupby('ticker')
                    .apply(lambda g: g['positive'].mean() - g['negative'].mean())
                    .rename('score').round(3))
portfolio_sentiment = ticker_sentiment.mean()

print('\n📊 Per-ticker sentiment:')
print(ticker_sentiment.to_string())
print(f'\n📰 Portfolio sentiment score: {portfolio_sentiment:+.3f}  (-1 bearish → +1 bullish)')

STOPS, stop_tag = build_stops(portfolio_sentiment)
print(f'\n🎯 Trailing stops ({stop_tag}):')
print(f'  {"Level":>10}   {"Exit fraction":>14}   Label')
print('  ' + '─'*42)
for s in STOPS:
    print(f'  {s["level"]:>+10.1%}   {s["exit_fraction"]:>14.1%}   {s["label"]}')


Fetching live headlines...
✅ 30 live headlines fetched.

Loading FinBERT (~420 MB on first run)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21691.33it/s]


,ticker,headline,positive,negative,neutral,sentiment
0,AAPL,"Despite Its Recent Partnership with Apple, MP ...",0.031,0.879,0.089,negative
1,AAPL,CDL’s $2.29 annual dividend beats Treasury yie...,0.942,0.029,0.029,positive
2,AAPL,"This ""Magnificent Seven"" Stock Is the Worst Pe...",0.049,0.456,0.495,neutral
3,AAPL,Is Intel’s (INTC) Confidential AI Push Quietly...,0.031,0.257,0.712,neutral
4,AAPL,The Microsoft Disaster: Why Dropping the OpenA...,0.023,0.493,0.485,negative
5,AAPL,Apple Inc. (AAPL)’s Durable Growth Narrative K...,0.933,0.013,0.054,positive
6,MSFT,2 Tech Stocks Down 25% That Smart Money Is Buy...,0.012,0.957,0.031,negative
7,MSFT,GlobalFoundries (GFS) Launches Dedicated Quant...,0.159,0.010,0.831,neutral
8,MSFT,Hyperscaler Debt Flood Brings Derivatives Bonanza,0.095,0.728,0.177,negative
9,MSFT,Paul Tudor Jones makes $8 billion bet on small...,0.070,0.020,0.911,neutral



📊 Per-ticker sentiment:
ticker
AAPL   -0.020
GS      0.035
JPM    -0.268
MSFT   -0.146
XOM     0.218

📰 Portfolio sentiment score: -0.036  (-1 bearish → +1 bullish)

🎯 Trailing stops (⚪ Neutral-derived):
       Level    Exit fraction   Label
  ──────────────────────────────────────────
       -5.0%            33.3%   Stop 1 (-5%)
      -10.0%            33.3%   Stop 2 (-10%)
      -15.0%            33.3%   Stop 3 (-15%)


---
## Phase 2 — Risk Models
### 2a · Download prices & build the portfolio return series

All risk analysis (vol, VaR, HMM) runs on the **weighted portfolio**, not a single stock.  
Switch weighting in the config cell: `'equal'`, `'manual'`, or `'optimised'`.


In [17]:
raw    = yf.download(TICKERS, start=START, end=END, auto_adjust=True)['Close']
prices = raw.dropna()
rets   = prices.pct_change().dropna()
print(f'Downloaded {len(prices)} days for {len(TICKERS)} tickers.')

# ── Resolve weights ───────────────────────────────────────────────────────
def resolve_weights(mode):
    n = len(TICKERS)
    if mode == 'equal':
        w = np.ones(n) / n
    elif mode == 'manual':
        w = np.array([MANUAL_WEIGHTS.get(t, 0.0) for t in TICKERS], dtype=float)
        assert abs(w.sum() - 1.0) < 1e-6, f'MANUAL_WEIGHTS must sum to 1 (got {w.sum():.4f})'
    elif mode == 'optimised':
        try:
            w = w_sharpe   # set by Phase 3
        except NameError:
            print('⚠️  w_sharpe not yet computed — falling back to equal weights.')
            print('   Run Phase 3 first, then re-run this cell.')
            w = np.ones(n) / n
    else:
        raise ValueError(f'Unknown PORTFOLIO_WEIGHTING: {mode!r}')
    return w

weights = resolve_weights(PORTFOLIO_WEIGHTING)
portfolio_rets = (rets * weights).sum(axis=1)   # weighted daily returns

print(f'\n📐 Portfolio weights ({PORTFOLIO_WEIGHTING}):')
for t, w in zip(TICKERS, weights):
    print(f'  {t}: {w:.1%}')
print(f'\nPortfolio return series: {len(portfolio_rets)} days')
print(f'  Annual mean : {portfolio_rets.mean()*252:.2%}')
print(f'  Annual vol  : {portfolio_rets.std()*np.sqrt(252):.2%}')


[*********************100%***********************]  5 of 5 completed

Downloaded 1605 days for 5 tickers.

📐 Portfolio weights (equal):
  AAPL: 20.0%
  MSFT: 20.0%
  XOM: 20.0%
  GS: 20.0%
  JPM: 20.0%

Portfolio return series: 1604 days
  Annual mean : 24.00%
  Annual vol  : 23.97%


### 2a-ii · Historical Portfolio Performance & Correlations

In [18]:

# ── Cumulative portfolio value (historical backtest) ──────────────────────
cumulative_port = (1 + portfolio_rets).cumprod() * PORT_VAL
cumulative_indiv = (1 + rets).cumprod() * PORT_VAL

fig = go.Figure()
for t in TICKERS:
    fig.add_trace(go.Scatter(x=cumulative_indiv.index, y=cumulative_indiv[t],
                             name=t, line=dict(width=1), opacity=0.6))
fig.add_trace(go.Scatter(x=cumulative_port.index, y=cumulative_port,
                         name=f'Portfolio ({PORTFOLIO_WEIGHTING})',
                         line=dict(width=3, color='white')))
fig.update_layout(title=f'Historical Cumulative Value — ${PORT_VAL:,.0f} starting capital',
                  xaxis_title='Date', yaxis_title='Portfolio Value ($)',
                  template='plotly_dark')
fig.show()

# ── Correlation heatmap ───────────────────────────────────────────────────
corr = rets.corr().round(2)
fig2 = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.index,
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    text=corr.values.round(2), texttemplate='%{text}',
    colorbar=dict(title='Correlation')))
fig2.update_layout(title='Return Correlation Matrix', template='plotly_dark')
fig2.show()

# ── Max drawdown per asset & portfolio ───────────────────────────────────
def max_drawdown(s):
    roll_max = s.cummax()
    dd = (s - roll_max) / roll_max
    return dd.min()

print('\n📉 Maximum Historical Drawdown:')
for t in TICKERS:
    cum = (1 + rets[t]).cumprod()
    print(f'  {t:6}  {max_drawdown(cum):.2%}')
print(f'  {"Portfolio":6}  {max_drawdown(cumulative_port):.2%}')



📉 Maximum Historical Drawdown:
  AAPL    -33.36%
  MSFT    -37.15%
  XOM     -54.99%
  GS      -45.62%
  JPM     -43.06%
  Portfolio  -38.79%


### 2b · Portfolio Volatility Forecasting (Rolling, EWMA, GARCH)

In [19]:
from arch import arch_model

r = portfolio_rets * 100   # scale for GARCH stability

roll_vol = r.rolling(ROLLING_WINDOW).std() * np.sqrt(252) / 100
ewma_vol = r.ewm(span=EWMA_SPAN).std()     * np.sqrt(252) / 100
garch_fit = arch_model(r, vol='Garch', p=GARCH_P, q=GARCH_Q, rescale=False).fit(disp='off')
garch_vol = garch_fit.conditional_volatility * np.sqrt(252) / 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=roll_vol.index, y=roll_vol,  name=f'Rolling {ROLLING_WINDOW}d'))
fig.add_trace(go.Scatter(x=ewma_vol.index, y=ewma_vol,  name=f'EWMA (span={EWMA_SPAN})'))
fig.add_trace(go.Scatter(x=garch_vol.index, y=garch_vol, name=f'GARCH({GARCH_P},{GARCH_Q})'))
fig.update_layout(title='Portfolio — Annualised Volatility Forecasts',
                  yaxis_tickformat='.0%', template='plotly_dark')
fig.show()
print(f'Latest GARCH portfolio vol: {garch_vol.iloc[-1]:.2%}')


Latest GARCH portfolio vol: 13.83%


### 2b-ii · Rolling Portfolio Metrics (Sharpe, Drawdown)

In [20]:

ROLLING_METRICS_WINDOW = 63   # ~1 quarter; override here if needed

roll_ret  = portfolio_rets.rolling(ROLLING_METRICS_WINDOW).mean() * 252
roll_vol2 = portfolio_rets.rolling(ROLLING_METRICS_WINDOW).std()  * np.sqrt(252)
roll_sharpe = (roll_ret - RISK_FREE) / roll_vol2

# Rolling drawdown on cumulative portfolio
cum_port  = (1 + portfolio_rets).cumprod()
roll_peak = cum_port.cummax()
roll_dd   = (cum_port - roll_peak) / roll_peak

fig = go.Figure()
fig.add_trace(go.Scatter(x=roll_sharpe.index, y=roll_sharpe,
                         name=f'Rolling {ROLLING_METRICS_WINDOW}d Sharpe',
                         line=dict(color='gold', width=2)))
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.add_hline(y=1, line_dash='dot',  line_color='lime', annotation_text='Sharpe = 1')
fig.update_layout(title=f'Portfolio Rolling {ROLLING_METRICS_WINDOW}-Day Sharpe Ratio',
                  yaxis_title='Sharpe', template='plotly_dark')
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=roll_dd.index, y=roll_dd,
                          fill='tozeroy', name='Drawdown',
                          line=dict(color='red', width=1)))
fig2.update_layout(title='Portfolio Historical Drawdown',
                   yaxis_title='Drawdown', yaxis_tickformat='.0%',
                   template='plotly_dark')
fig2.show()

print(f'Rolling {ROLLING_METRICS_WINDOW}d Sharpe  — current: {roll_sharpe.iloc[-1]:.2f} '
      f'| max: {roll_sharpe.max():.2f} | min: {roll_sharpe.min():.2f}')
print(f'Max historical drawdown (portfolio): {roll_dd.min():.2%}')
print(f'Current drawdown from peak:          {roll_dd.iloc[-1]:.2%}')


Rolling 63d Sharpe  — current: 1.77 | max: 6.46 | min: -3.60
Max historical drawdown (portfolio): -38.79%
Current drawdown from peak:          0.00%


### 2c · Portfolio VaR & CVaR

In [21]:
from scipy import stats

def compute_risk(returns, confidence_levels=VAR_CONFIDENCE_LEVELS):
    rows, mu, sigma = [], returns.mean(), returns.std()
    sim = np.random.normal(mu, sigma, VAR_SIM_SIZE)
    for cl in confidence_levels:
        a   = 1 - cl
        hv  = -np.percentile(returns, a*100)
        hcv = -returns[returns <= -hv].mean()
        pv  = -(mu + stats.norm.ppf(a)*sigma)
        pcv = -(mu - sigma*stats.norm.pdf(stats.norm.ppf(a))/a)
        mv  = -np.percentile(sim, a*100)
        mcv = -sim[sim <= -mv].mean()
        rows.append({'Confidence': f'{cl:.0%}',
                     'Hist VaR': f'{hv:.2%}',  'Hist CVaR': f'{hcv:.2%}',
                     'Param VaR': f'{pv:.2%}', 'Param CVaR': f'{pcv:.2%}',
                     'MC VaR': f'{mv:.2%}',    'MC CVaR': f'{mcv:.2%}'})
    return pd.DataFrame(rows)

display(compute_risk(portfolio_rets))


,Confidence,Hist VaR,Hist CVaR,Param VaR,Param CVaR,MC VaR,MC CVaR
0,95%,2.03%,3.46%,2.39%,3.02%,2.37%,3.01%
1,99%,4.08%,6.33%,3.42%,3.93%,3.41%,3.91%


### 2d · Portfolio Regime Detection (HMM)

In [22]:
from hmmlearn.hmm import GaussianHMM

if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

r_arr   = portfolio_rets.values.reshape(-1, 1)
hmm     = GaussianHMM(n_components=HMM_N_REGIMES, covariance_type='full',
                      n_iter=HMM_N_ITER, random_state=RANDOM_SEED)
hmm.fit(r_arr)
regimes = hmm.predict(r_arr)
ranking = sorted(range(HMM_N_REGIMES), key=lambda i: hmm.means_[i][0])

regime_name_map = {}
for pos, state in enumerate(ranking):
    if pos == 0:
        regime_name_map[state] = 'Bear 🔴'
    elif pos == len(ranking) - 1:
        regime_name_map[state] = 'Bull 🟢'
    else:
        regime_name_map[state] = 'Neutral ⚪'

regime_labels = pd.Series([regime_name_map[r] for r in regimes], index=portfolio_rets.index)

fig = px.scatter(x=portfolio_rets.index, y=portfolio_rets, color=regime_labels,
                 color_discrete_map={'Bear 🔴':'red','Neutral ⚪':'grey','Bull 🟢':'green'},
                 title=f'Portfolio Daily Returns — HMM Regime Detection ({HMM_N_REGIMES} regimes)',
                 template='plotly_dark')
fig.update_traces(marker_size=3)
fig.show()

current_regime = regime_labels.iloc[-1]
print(f'Current regime: {current_regime}')

print('\n📊 Regime summary:')
for name in ['Bull 🟢', 'Neutral ⚪', 'Bear 🔴']:
    mask = regime_labels == name
    if mask.any():
        r = portfolio_rets[mask]
        print(f'  {name:15}  {mask.mean():.1%} of days  '
              f'| mean ret {r.mean()*252:>+.1%}/yr  '
              f'| vol {r.std()*np.sqrt(252):.1%}/yr')


Current regime: Bull 🟢

📊 Regime summary:
  Bull 🟢           43.5% of days  | mean ret +37.8%/yr  | vol 16.9%/yr
  Neutral ⚪        43.5% of days  | mean ret +14.9%/yr  | vol 15.1%/yr
  Bear 🔴           13.0% of days  | mean ret +8.2%/yr  | vol 52.1%/yr


---
## Phase 3 — Portfolio Optimisation (Efficient Frontier)

Finds the **max-Sharpe** portfolio using your tickers and sentiment-adjusted expected returns.

> 💡 To use these weights in Phase 2, set `PORTFOLIO_WEIGHTING = 'optimised'` in the config  
> and re-run **Phase 2a onward**.


In [23]:
from scipy.optimize import minimize

mu_annual  = rets.mean() * 252
cov_annual = rets.cov()  * 252
n          = len(TICKERS)

for t in TICKERS:
    if t in ticker_sentiment.index:
        mu_annual[t] += ticker_sentiment[t] * SENTIMENT_RETURN_BOOST

def portfolio_stats(w):
    ret    = w @ mu_annual
    vol    = np.sqrt(w @ cov_annual @ w)
    sharpe = (ret - RISK_FREE) / vol
    return ret, vol, sharpe

opt      = minimize(lambda w: -portfolio_stats(w)[2],
                    np.ones(n)/n, method='SLSQP',
                    bounds=[(0,1)]*n,
                    constraints={'type':'eq','fun':lambda w: w.sum()-1})
w_sharpe = opt.x

sim_w     = np.random.dirichlet(np.ones(n), N_SIM_EF)
sim_stats = np.array([portfolio_stats(w) for w in sim_w])

fig = go.Figure()
fig.add_trace(go.Scatter(x=sim_stats[:,1], y=sim_stats[:,0], mode='markers',
    marker=dict(color=sim_stats[:,2], colorscale='Viridis', size=4,
                showscale=True, colorbar=dict(title='Sharpe')),
    name=f'{N_SIM_EF} random portfolios'))
r_opt, v_opt, s_opt = portfolio_stats(w_sharpe)
fig.add_trace(go.Scatter(x=[v_opt], y=[r_opt], mode='markers+text',
    marker=dict(color='red', size=14, symbol='star'),
    text=['Max Sharpe'], textposition='top right', name='Max Sharpe'))
fig.update_layout(title='Efficient Frontier (sentiment-enhanced returns)',
                  xaxis_title='Volatility', yaxis_title='Return',
                  xaxis_tickformat='.0%', yaxis_tickformat='.0%',
                  template='plotly_dark')
fig.show()

print('\n📌 Max-Sharpe weights:')
for t, w in zip(TICKERS, w_sharpe):
    print(f'  {t}: {w:.1%}')
print(f'\n  Return: {r_opt:.2%}  |  Vol: {v_opt:.2%}  |  Sharpe: {s_opt:.2f}')
print('\n💡 To use these weights everywhere, set PORTFOLIO_WEIGHTING = \'optimised\' and re-run from Phase 2a.')



📌 Max-Sharpe weights:
  AAPL: 40.8%
  MSFT: 41.9%
  XOM: 0.0%
  GS: 0.0%
  JPM: 17.3%

  Return: 27.83%  |  Vol: 25.83%  |  Sharpe: 0.88

💡 To use these weights everywhere, set PORTFOLIO_WEIGHTING = 'optimised' and re-run from Phase 2a.


---
## Phase 4 — Regime-Aware Monte Carlo with Trailing Stops

GBM drift & vol come from the **current HMM regime** of the portfolio.  
The 3-tranche trailing stop engine runs day-by-day on each simulated path.


In [24]:
if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

rp    = REGIME_PARAMS.get(current_regime,
        REGIME_PARAMS.get('Neutral ⚪', list(REGIME_PARAMS.values())[1]))
drift = rp['drift']/252          + portfolio_sentiment * SENTIMENT_DRIFT_NUDGE
vol   = rp['vol']  /np.sqrt(252)

shocks     = np.random.normal(0, 1, (HORIZON, N_PATHS))
log_ret    = (drift - 0.5*vol**2) + vol*shocks
daily_rets = np.exp(log_ret)

paths_raw = np.vstack([np.full(N_PATHS, float(PORT_VAL)),
                       PORT_VAL * np.exp(np.cumsum(log_ret, axis=0))])

stops_sorted = sorted(STOPS, key=lambda s: s['level'])
in_market    = np.full(N_PATHS, float(PORT_VAL))
cash         = np.zeros(N_PATHS)
peak         = np.full(N_PATHS, float(PORT_VAL))
triggered    = [np.zeros(N_PATHS, dtype=bool) for _ in stops_sorted]
trigger_day  = [np.full(N_PATHS, -1)          for _ in stops_sorted]
paths_stopped    = np.zeros((HORIZON+1, N_PATHS))
paths_stopped[0] = PORT_VAL

for day in range(1, HORIZON+1):
    in_market *= daily_rets[day-1]
    peak       = np.maximum(peak, in_market)
    drawdown   = np.where(peak > 0, (in_market-peak)/peak, 0.0)
    for i, stop in enumerate(stops_sorted):
        fire = (~triggered[i]) & (drawdown <= stop['level'])
        if fire.any():
            locked = stop['exit_fraction'] * in_market[fire]
            cash[fire] += locked; in_market[fire] -= locked
            triggered[i][fire] = True; trigger_day[i][fire] = day
    paths_stopped[day] = in_market + cash

final_raw, final_stopped = paths_raw[-1], paths_stopped[-1]
var95_raw     = np.percentile(final_raw,     5)
var95_stopped = np.percentile(final_stopped, 5)
days_ax       = np.arange(HORIZON + 1)
pct           = lambda arr, q: np.percentile(arr, q, axis=1)

last_stop = np.zeros(N_PATHS, dtype=int)
for i in range(len(stops_sorted)): last_stop[triggered[i]] = i+1

STOP_COLOURS = {0:'steelblue', 1:'#f0e68c', 2:'#ffa500', 3:'#ff4444'}
STOP_NAMES   = {0:'No stop hit',
                **{i+1: stops_sorted[i]['label'] for i in range(len(stops_sorted))}}

fig = go.Figure()
already_shown = set()
for i in np.random.choice(N_PATHS, PLOT_SAMPLE_PATHS, replace=False):
    grp = last_stop[i]; name = STOP_NAMES[grp]
    show = name not in already_shown; already_shown.add(name)
    fig.add_trace(go.Scatter(x=days_ax, y=paths_stopped[:,i],
        line=dict(width=0.5, color=STOP_COLOURS[grp]),
        name=name, legendgroup=name, showlegend=show, opacity=0.5))

for q,col,dash,lbl in [(95,'lime','dot','95th (w/ stops)'),
                        (50,'white','solid','Median (w/ stops)'),
                        (5,'red','dot','5th (w/ stops)')]:
    fig.add_trace(go.Scatter(x=days_ax, y=pct(paths_stopped,q),
                             line=dict(color=col,width=2,dash=dash), name=lbl))
for q,lbl in [(50,'Median (no stops)'),(5,'5th (no stops)')]:
    fig.add_trace(go.Scatter(x=days_ax, y=pct(paths_raw,q),
                             line=dict(color='grey',width=1.5,dash='dash'), name=lbl))

for i, stop in enumerate(stops_sorted):
    fired = triggered[i]
    if fired.any():
        avg_day = trigger_day[i][fired].mean()
        fig.add_vline(x=avg_day, line_dash='dot', line_color=STOP_COLOURS[i+1], line_width=1.5,
                      annotation_text=f"{stop['label']}<br>avg day {avg_day:.0f} ({fired.mean():.0%})",
                      annotation_font_size=10)

fig.update_layout(
    title=(f'Monte Carlo — {N_PATHS:,} paths | Regime: {current_regime} '
           f'| Weights: {PORTFOLIO_WEIGHTING} | Sentiment: {portfolio_sentiment:+.3f}'),
    xaxis_title='Trading Days', yaxis_title='Portfolio Value ($)',
    template='plotly_dark', height=600)
fig.show()

print(f'\n📉 1-Year Risk Summary | Weights: {PORTFOLIO_WEIGHTING} | Regime: {current_regime}')
print(f'  {"":28}  {"No stops":>14}  {"With stops":>14}')
print('  ' + '─'*60)
for label, q in [('Median final value',50),('95th pct',95),('5th pct',5)]:
    print(f'  {label:28}  ${np.percentile(final_raw,q):>12,.0f}  '
          f'${np.percentile(final_stopped,q):>12,.0f}')
print(f'  {"VaR 95%":28}  ${PORT_VAL-var95_raw:>12,.0f}  ${PORT_VAL-var95_stopped:>12,.0f}')

print('\n📊 Trailing stop trigger rates:')
for i, stop in enumerate(stops_sorted):
    fired = triggered[i]
    avg_d = trigger_day[i][fired].mean() if fired.any() else float('nan')
    status = (f'triggered {fired.mean():.1%} of paths | avg day {avg_d:.0f}'
              if fired.any() else 'never triggered')
    print(f"  {stop['label']:20}  {status}")



📉 1-Year Risk Summary | Weights: equal | Regime: Bull 🟢
                                      No stops      With stops
  ────────────────────────────────────────────────────────────
  Median final value            $   1,116,952  $   1,042,554
  95th pct                      $   1,358,495  $   1,270,465
  5th pct                       $     917,279  $     950,964
  VaR 95%                       $      82,721  $      49,036

📊 Trailing stop trigger rates:
  Stop 3 (-15%)         triggered 95.5% of paths | avg day 78
  Stop 2 (-10%)         triggered 95.5% of paths | avg day 78
  Stop 1 (-5%)          triggered 95.5% of paths | avg day 77


---
## Stress Test — What the scenarios mean & how to set them

Each scenario tweaks two parameters of the GBM that drives the Monte Carlo:

| Parameter | What it represents | Example |
|---|---|---|
| `drift_shock` | **Annual return penalty** added on top of the base regime drift. Negative = bad news. | `-0.10` = "this shock costs the portfolio 10% annual return" |
| `vol_mult` | **Volatility multiplier** relative to base regime vol. >1 = more turbulent. | `2.0` = "twice as volatile as normal" |

### How to calibrate them from historical events

A useful approach: look at what actually happened during a comparable past episode.

**Drift shock** — take the annualised excess return of your portfolio during the event vs a calm baseline period, e.g.:
- 2022 rate-hike cycle hit a balanced equity portfolio by roughly **−15 to −20% annual return** → `drift_shock = -0.15`
- A sector-specific shock (oil names during 2014−16 oil crash) might be **−30 to −40%** → `drift_shock = -0.35`
- A mild macro headwind might only cost **−3 to −5%** → `drift_shock = -0.03`

**Vol multiplier** — compare the average VIX (or realised vol) during the episode to the calm period:
- VIX doubles from 15 → 30: `vol_mult ≈ 2.0`
- VIX goes from 15 → 45 (2020 COVID crash): `vol_mult ≈ 3.0`
- Mild turbulence (VIX 15 → 20): `vol_mult ≈ 1.3`

### Suggested starting points by event type

```python
# Add / edit entries in SCENARIOS in the config cell
SCENARIOS = {
    'Base case':           {'drift_shock':  0.00, 'vol_mult': 1.0},
    'Rate hike +100 bps':  {'drift_shock': -0.03, 'vol_mult': 1.2},
    'Mild recession':      {'drift_shock': -0.10, 'vol_mult': 1.5},
    'Deep recession':      {'drift_shock': -0.20, 'vol_mult': 2.0},
    'Oil shock −20%':      {'drift_shock': -0.06, 'vol_mult': 1.4},
    'Credit crunch':       {'drift_shock': -0.15, 'vol_mult': 2.2},
    'COVID-style crash':   {'drift_shock': -0.30, 'vol_mult': 3.0},
    'Soft landing':        {'drift_shock':  0.03, 'vol_mult': 0.8},
}
```


In [25]:
rows = []
for name, params in SCENARIOS.items():
    d  = drift + params['drift_shock'] / 252
    v  = vol   * params['vol_mult']
    lr = (d - 0.5*v**2) + v * np.random.normal(0, 1, (HORIZON, N_PATHS_STRESS))
    dr = np.exp(lr)

    im = np.full(N_PATHS_STRESS, float(PORT_VAL))
    cs = np.zeros(N_PATHS_STRESS)
    pk = np.full(N_PATHS_STRESS, float(PORT_VAL))
    trig = [np.zeros(N_PATHS_STRESS, dtype=bool) for _ in stops_sorted]

    for day in range(HORIZON):
        im *= dr[day]; pk = np.maximum(pk, im)
        dd = np.where(pk > 0, (im-pk)/pk, 0.0)
        for i, stop in enumerate(stops_sorted):
            fire = (~trig[i]) & (dd <= stop['level'])
            if fire.any():
                locked = stop['exit_fraction'] * im[fire]
                cs[fire] += locked; im[fire] -= locked; trig[i][fire] = True

    f = im + cs
    row = {'Scenario': name,
           'Drift shock': f"{params['drift_shock']:>+.0%}/yr",
           'Vol mult':    f"{params['vol_mult']:.1f}×",
           'Median P&L':  f'${np.median(f)-PORT_VAL:>+,.0f}',
           '5th pct P&L': f'${np.percentile(f,5)-PORT_VAL:>+,.0f}',
           'Prob loss':   f'{(f<PORT_VAL).mean():.1%}'}
    for i, stop in enumerate(stops_sorted):
        row[stop['label']] = f'{trig[i].mean():.1%}'
    rows.append(row)

display(pd.DataFrame(rows))


,Scenario,Drift shock,Vol mult,Median P&L,5th pct P&L,Prob loss,Stop 3 (-15%),Stop 2 (-10%),Stop 1 (-5%)
0,Base case,+0%/yr,1.0×,"$+42,164","$-48,609",26.3%,95.6%,95.6%,95.7%
1,Oil −20%,-6%/yr,1.4×,"$+9,763","$-88,954",44.9%,100.0%,100.0%,100.0%
2,Rates +100 bps,-3%/yr,1.2×,"$+22,779","$-71,030",36.8%,99.5%,99.5%,99.6%
3,Market crash −30%,-25%/yr,2.5×,"$-57,519","$-174,812",71.0%,100.0%,100.0%,100.0%


---
## 📋 Final Summary Dashboard
All key outputs in one place.

In [26]:

print('=' * 65)
print('  📊 TRADING RISK MANAGER — FINAL SUMMARY')
print('=' * 65)

print(f'\n🗂  Portfolio  ({PORTFOLIO_WEIGHTING} weights, {START} → {END})')
for t, w in zip(TICKERS, weights):
    sent = ticker_sentiment.get(t, float('nan'))
    print(f'   {t:6} {w:>6.1%}   sentiment: {sent:>+.3f}')
print(f'   Portfolio sentiment: {portfolio_sentiment:+.3f}')

print(f'\n📈 Historical Performance')
ann_ret = portfolio_rets.mean() * 252
ann_vol = portfolio_rets.std()  * np.sqrt(252)
print(f'   Annual return  : {ann_ret:>+.2%}')
print(f'   Annual vol     : {ann_vol:.2%}')
print(f'   Sharpe ratio   : {(ann_ret - RISK_FREE) / ann_vol:.2f}')
print(f'   Max drawdown   : {roll_dd.min():.2%}')

print(f'\n🎯 Risk Metrics (portfolio, GARCH vol = {garch_vol.iloc[-1]:.2%})')
risk_tbl = compute_risk(portfolio_rets)
print(risk_tbl.to_string(index=False))

print(f'\n🔭 Current Regime  : {current_regime}')
print(f'   Trailing stops  : {stop_tag}')
for s in STOPS:
    fired_i = next((triggered[i] for i, st in enumerate(stops_sorted)
                    if st["label"] == s["label"]), None)
    rate = f'{fired_i.mean():.1%} of MC paths' if fired_i is not None else '—'
    print(f'   {s["label"]:20} {s["level"]:>+.0%} | triggered in {rate}')

print(f'\n💼 Optimised Max-Sharpe Portfolio')
for t, w in zip(TICKERS, w_sharpe):
    print(f'   {t:6} {w:.1%}')
print(f'   Expected return: {r_opt:.2%}  |  Vol: {v_opt:.2%}  |  Sharpe: {s_opt:.2f}')

print(f'\n🎲 Monte Carlo ({N_PATHS:,} paths, {HORIZON}d, ${PORT_VAL:,.0f} start)')
print(f'   {"":26}  {"No stops":>12}  {"With stops":>12}')
print('   ' + '─' * 54)
for label, q in [('Median final value', 50), ('95th pct', 95), ('5th pct', 5)]:
    print(f'   {label:26}  ${np.percentile(final_raw,q):>10,.0f}  '
          f'${np.percentile(final_stopped,q):>10,.0f}')
print(f'   {"VaR 95%":26}  ${PORT_VAL-var95_raw:>10,.0f}  '
      f'${PORT_VAL-var95_stopped:>10,.0f}')
print('=' * 65)


  📊 TRADING RISK MANAGER — FINAL SUMMARY

🗂  Portfolio  (equal weights, 2020-01-01 → 2026-05-22)
   AAPL    20.0%   sentiment: -0.020
   MSFT    20.0%   sentiment: -0.146
   XOM     20.0%   sentiment: +0.218
   GS      20.0%   sentiment: +0.035
   JPM     20.0%   sentiment: -0.268
   Portfolio sentiment: -0.036

📈 Historical Performance
   Annual return  : +24.00%
   Annual vol     : 23.97%
   Sharpe ratio   : 0.79
   Max drawdown   : -38.79%

🎯 Risk Metrics (portfolio, GARCH vol = 13.83%)
Confidence Hist VaR Hist CVaR Param VaR Param CVaR MC VaR MC CVaR
       95%    2.03%     3.46%     2.39%      3.02%  2.39%   3.03%
       99%    4.08%     6.33%     3.42%      3.93%  3.44%   3.95%

🔭 Current Regime  : Bull 🟢
   Trailing stops  : ⚪ Neutral-derived
   Stop 1 (-5%)         -5% | triggered in 95.5% of MC paths
   Stop 2 (-10%)        -10% | triggered in 95.5% of MC paths
   Stop 3 (-15%)        -15% | triggered in 95.5% of MC paths

💼 Optimised Max-Sharpe Portfolio
   AAPL   40.8%
   MS